### TCGA / GDC Portal

https://portal.gdc.cancer.gov/

## GDC

https://docs.gdc.cancer.gov/API/Users_Guide/Downloading_Files/

### 🧠 Big picture (this is powerful)

- You’ve essentially built the backbone of:
- cBioPortal-like explorer
- cohort stratification engine
- patient-specific pipeline (your perturbation_agent 🔥)

### 💡 Flow

> project → project_id  
> primary_sites → pid and disease_type  
> subtypes → subtype_id  
> stages →  stage_id  
> case_id → case_ids or UUIDs  
> samples → sample_id [tumor, normal]  
> aliquots → aliquot_id  
> files → file_id  


### 👉 a TCGA cohort builder + data retrieval engine

Which means you can easily extend to:

> pan-cancer analyses  
> patient-specific pipelines (your perturb_agent 🔥)  
> automated workflows (Nextflow-ready)  

#### 🚀 Next steps

> Production pipeline  
  - CLI tool
  - cached queries
  - reproducible outputs

> Expression matrix builder  
  - auto-download
  - merge TSVs
  - DESeq2-ready matrix

> Patient-level analysis
  - per-case DEG
  - pathway perturbation scoring

In [ ]:
import os, sys
import pandas as pd

from pathlib import Path

ROOT = Path().resolve().parent.parent
SRC = os.path.join(ROOT, "src")

if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

print("ROOT:", ROOT)
print("SRC added:", SRC)

from libs.tcga_gdc_lib import *
from libs.Basic import *


### Defaults

In [ ]:
ROOT = Path().resolve().parent
root_data = os.path.join(ROOT, "data/tcga")

gdc = GDC(root_data=root_data)

os.listdir(root_data)


### All programs

In [ ]:
force=False
verbose=True

prog_list = gdc.list_gdc_progams(force=force, verbose=verbose)

# prog_list

### Get valid subtypes

facets = "diagnoses.primary_diagnosis"

### For each subtype → get stages

facets = "diagnoses.tumor_stage"

### For each (subtype, stage) → get samples

fields = "case_id,submitter_id,diagnoses.tumor_stage"

### Then → fetch RNA-seq files

endpoint = "/files"
field = "cases.case_id"



In [ ]:
force=False
verbose=True

program = 'TCGA'

dfc = gdc.get_primary_sites(program=program, force=force, verbose=verbose)

dfc.head(3)


### Subtypes

In [ ]:
force=False
verbose=True

i=1
pid = dfc.iloc[i].project_id
primary_site = dfc.iloc[i].primary_site

print(pid, primary_site)

df_subt = gdc.build_subtypes(pid=pid, do_filter=True, force=force, verbose=verbose)

print(len(df_subt))
df_subt

### Stages

In [ ]:
force=False
verbose=True

i=0
subtype = df_subt.iloc[i].subtype_raw

print(pid, subtype)

df_stage = gdc.build_stages(pid=pid, subtype=subtype, do_filter=True, force=force, verbose=verbose)

print(len(df_stage))
df_stage

In [ ]:
force=False
verbose=True

i=0
stage = df_stage.iloc[i].stage_raw


print(pid, subtype, stage)

df_case = gdc.build_cases(pid=pid, subtype=subtype, stage=stage, force=force, verbose=verbose)
print(len(df_case))
df_case.head(12)

### Given a PS, Subtype, and Stage --> return the SAMPLES

In [ ]:
force=False
verbose=True

print(pid, subtype, stage)

df_sample = gdc.get_samples(pid=pid, subtype=subtype, stage=stage, batch_size=200, force=force, verbose=verbose)
print(len(df_sample))
df_sample.head(8)

In [ ]:
force=False
verbose=True

case_ids = list(np.unique(df_sample.case_id))

print(pid, subtype, stage, f"cases {len(case_ids)}")

df_files = gdc.get_expression_files_given_samples(pid=pid, subtype=subtype, stage=stage, 
                                                  case_ids=case_ids, batch_size=20, force=force, verbose=verbose)
print(len(df_files))
df_files.head(6)

In [ ]:
case_ids[:10]